# Lesson 14 — Storage & Access Internals

> **Prerequisite:** Tiers 1–4, especially lesson 12 (indexes) and 13 (execution plans). This lesson explains the *why* under those topics.

## What you'll learn
- The 8KB page: header, slot array, and rows
- Extents (mixed vs uniform) and how tables are allocated
- Heaps vs clustered B-trees, and how a row is physically located
- Reading real pages with `DBCC PAGE` and the allocation DMVs
- Why page splits, fragmentation, and heap forwarded records happen

## Setup
Run `setup.sql` once. It creates the `lesson14` schema with a heap copy and a clustered copy of `SalesOrderHeader`, plus a small `SplitDemo` table for page-split experiments. Re-run to reset.

**Tooling note:** run the SQL cells in this notebook with Azure Data Studio connected to the mssql-learn container (attach the SQL kernel to your saved localhost connection). `DBCC PAGE` needs a real FileId/PageId, so its line in Example 3 is left commented out — read the page id from the previous example's output, then uncomment and edit the `DBCC PAGE (...)` line to run it. `DBCC PAGE` is read-only, but it is a diagnostic tool, not a production API.

In [ ]:
USE AdventureWorks2022;
GO

## Concepts

### The 8KB page
Everything in SQL Server is stored in 8KB pages. A page has a 96-byte header, then rows growing from the top, and a slot array growing from the bottom that points to each row. Eight contiguous pages form an **extent** (64KB).

### Heap vs clustered index
- A **heap** has no clustered index — rows live wherever they fit, located via a Row ID (file:page:slot).
- A **clustered index** sorts the data pages by the key in a B-tree (root → intermediate → leaf). The leaf level *is* the data.

### How a row is located
`%%physloc%%` exposes a row's physical RID; `sys.fn_PhysLocFormatter` decodes it to `(file:page:slot)`. `sys.dm_db_database_page_allocations` lists every page of an object. `DBCC PAGE` (with trace flag 3604) prints a page's raw contents.

### Why things go wrong
- **Page split:** inserting into a full page forces SQL Server to allocate a new page and move ~half the rows — causing fragmentation and extra logging.
- **Fragmentation:** logical page order drifts from physical order; scans do more random I/O. `sys.dm_db_index_physical_stats` measures it.
- **Forwarded records:** in a *heap*, an updated row that no longer fits leaves a forwarding pointer to its new location — an extra I/O on every read.

## Worked investigations

### Example 1 — Find the physical location (file:page:slot) of specific rows

`%%physloc%%` — the physical address of a row.

In [ ]:
-- %%physloc%% is an undocumented-but-stable virtual column; fn_PhysLocFormatter decodes it.
SELECT TOP 5
    SalesOrderID,
    sys.fn_PhysLocFormatter(%%physloc%%) AS PhysicalLocation   -- format: (file:page:slot)
FROM lesson14.SalesOrderCI
ORDER BY SalesOrderID;

### Example 2 — List the pages allocated to a table and their types

`sys.dm_db_database_page_allocations` — list a table's pages and types.

In [ ]:
-- page_type_desc shows DATA_PAGE, INDEX_PAGE, IAM_PAGE, etc.
SELECT
    allocated_page_file_id  AS FileId,
    allocated_page_page_id  AS PageId,
    page_type_desc,
    index_id
FROM sys.dm_db_database_page_allocations(
        DB_ID(), OBJECT_ID('lesson14.SalesOrderCI'), NULL, NULL, 'DETAILED')
WHERE page_type_desc IS NOT NULL
ORDER BY page_type_desc, PageId;

### Example 3 — Crack open a real data page with DBCC PAGE

`DBCC PAGE` — crack open a real data page. Trace flag 3604 redirects DBCC output to the client instead of the error log. The `DBCC PAGE (...)` line stays commented — substitute a real DATA_PAGE PageId from Example 2, then uncomment and run it.

In [ ]:
-- Trace flag 3604 redirects DBCC output to the client instead of the error log.
-- Pick any DATA_PAGE PageId from Example 2 and substitute it below.
DBCC TRACEON(3604);
-- DBCC PAGE ('AdventureWorks2022', <FileId>, <PageId>, 3);  -- style 3 = page header + each row
-- Example with a placeholder page number — replace 0 with a real PageId from Example 2:
-- DBCC PAGE ('AdventureWorks2022', 1, 0, 3);
PRINT 'Replace the FileId/PageId above with a real DATA_PAGE from Example 2, then run DBCC PAGE.';
DBCC TRACEOFF(3604);

### Example 4 — Fragmentation and page density of the clustered index

`sys.dm_db_index_physical_stats` — fragmentation and page fullness.

In [ ]:
SELECT
    i.name                              AS IndexName,
    ips.index_type_desc,
    ips.page_count,
    ips.avg_fragmentation_in_percent,
    ips.avg_page_space_used_in_percent  AS AvgPageFullnessPct,
    ips.forwarded_record_count          -- only meaningful for heaps
FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SalesOrderCI'), NULL, NULL, 'DETAILED') AS ips
JOIN sys.indexes AS i
  ON i.object_id = ips.object_id AND i.index_id = ips.index_id;

### Example 5 — Heaps create FORWARDED RECORDS when an updated row no longer fits in place

Heap forwarded records — before/after an in-place update.

In [ ]:
-- Inspect the heap before the update
SELECT 'before' AS Phase, forwarded_record_count, page_count
FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SalesOrderHeap'), 0, NULL, 'DETAILED');

-- Widen rows in place by filling a variable-length column that starts out NULL.
-- Comment is nvarchar(128 chars); seeding it grows the row so it no longer fits in place.
UPDATE lesson14.SalesOrderHeap
SET Comment = REPLICATE(N'Z', 120)
WHERE SalesOrderID % 5 = 0;

-- Inspect again — forwarded_record_count rises because moved rows leave a forwarding pointer
SELECT 'after' AS Phase, forwarded_record_count, page_count
FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SalesOrderHeap'), 0, NULL, 'DETAILED');

### Example 6 — Trigger and observe a PAGE SPLIT on a clustered B-tree

Page split — before/after a middle-key insert.

In [ ]:
-- Page count BEFORE inserting a middle key
SELECT 'before split' AS Phase, page_count, avg_page_space_used_in_percent
FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SplitDemo'), 1, NULL, 'DETAILED');

-- Insert a key that lands in the MIDDLE of a full page (rows are 2000 bytes; ~4 fit per 8KB page)
INSERT lesson14.SplitDemo (Id) VALUES (15), (25), (35);

-- Page count AFTER — the engine splits full pages to make room, raising page_count
SELECT 'after split' AS Phase, page_count, avg_page_space_used_in_percent
FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SplitDemo'), 1, NULL, 'DETAILED');

## Common issues this explains
- Why heaps can be slow to read after updates (forwarding pointers).
- Why random-key inserts (e.g., GUIDs) fragment a clustered index.
- Why a low fill factor trades space for fewer splits.

## Pitfalls
- `DBCC PAGE` is a learning/diagnostic tool, not a documented production API — don't build on it.
- `'LIMITED'` mode of `dm_db_index_physical_stats` does not report `forwarded_record_count` or page fullness; use `'DETAILED'` (slower) when you need them.
- `%%physloc%%` changes when a row moves — it is not a stable key.

## Cheatsheet
See `cheatsheets/05-indexes.md`

## Exercises
Open `exercises.ipynb` and work them in order. Solutions in `exercises-solutions.ipynb`.